# AIE S3 — Bike Sharing Demand: Modelling & Evaluation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/data_science_practice/blob/main/website/public/modules/python-ai-engineering/challenges/aie-s3-bike-demand.ipynb)

**Regression.** The same hourly bike-rental data as Session 2, and
the same challenge.

Challenge: <https://ml-arena.com/viewchallenge/183>

---

In Session 2 you looked at this table and fitted one straight line
through it. That model scored **−138.88** on the leaderboard. This
notebook is what comes next: the data does not change, the model
does — and so does the machinery that tells you *which* model to
keep, before the leaderboard tells you.

Four steps, and their order is the lesson:

1. **split first** — a test slice you spend once, a validation slice
   you tune against;
2. **transform second**, with every statistic learned on the training
   rows only;
3. **compare models** on training *and* validation, and watch the two
   numbers disagree;
4. **search the hyperparameters** with `GridSearchCV`, then spend the
   test set.

It ends at about **−48** on the leaderboard, from −138.88, and not
one new feature is engineered on the way.

---

## 0. Setup

The distribution is **`mlarena-sdk`** and it imports as `mlarena`.
`pip install mlarena` is an unrelated package by another author.

In [ ]:
!pip install -q mlarena-sdk

---

## 1. Get the data

Your key is on your ML-Arena **Profile** page. Same three files as
Session 2 — this is the same challenge.

In [ ]:
import mlarena

API_KEY = "mlk_user_..."   # <- paste yours here
CHALLENGE_ID = 183

client = mlarena.connect(api_key=API_KEY)
client.download_dataset(CHALLENGE_ID, ".")

---

## 2. Read it, and name it for what it is

In [ ]:
import pandas as pd

X = pd.read_csv("X_train.csv")               # every row you have a label for
y = pd.read_csv("y_train.csv")["prediction"]
X_submission = pd.read_csv("X_test.csv")     # the rows the leaderboard scores

print("X", X.shape, " y", y.shape, " X_submission", X_submission.shape)
X.head()

The rename is deliberate, and it is the first thing this session
changes.

The files on disk are still called `X_train.csv` and `X_test.csv`.
But from here on **"test" means the slice of your own data you keep
back for the final number** — and you are about to build one. Two
different things cannot share a name inside the same notebook, so:

| variable | what it is | can you score it? |
|---|---|---|
| `X`, `y` | all 13,903 labelled hours | yes — they are yours to split |
| `X_submission` | 3,476 hours with the labels held back | no, only the leaderboard can |

Nothing on disk moved. Only the words did.

---

## 3. Split first — before you touch a single value

The split comes before the imputation, before the encoding, before
the first `fit`. Do it in the other order and the median you fill
with has already read the rows you were about to score yourself on.

**And this split has to respect time.** `X` ships in calendar order,
and the challenge holds out the *last* fifth of the two years — so
the real task is to forecast forward. A shuffled split would let the
model train on 21:00 and be scored on 20:00 of the same evening,
which is not the task and is worth about 35 MAE of flattery.

So: 20% off the end for test, then 20% off the end of what is left
for validation. Three slices, in time order.

In [ ]:
n_test = int(0.2 * len(X))
X_pool,  y_pool  = X.iloc[:-n_test], y.iloc[:-n_test]
X_test,  y_test  = X.iloc[-n_test:], y.iloc[-n_test:]

n_val = int(0.2 * len(X_pool))
X_train, y_train = X_pool.iloc[:-n_val], y_pool.iloc[:-n_val]
X_val,   y_val   = X_pool.iloc[-n_val:], y_pool.iloc[-n_val:]

print("train", X_train.shape, " val", X_val.shape, " test", X_test.shape)
print("target mean — train %.1f  val %.1f  test %.1f"
      % (y_train.mean(), y_val.mean(), y_test.mean()))

`8,899 / 2,224 / 2,780`. Note the target means: **143 / 184 / 268**.
The three slices are not samples of one distribution — the system
grew over the two years, so each slice is genuinely harder than the
one before it. That is what a forecast is, and it is why a shuffled
split flatters.

What each slice is for, in one line each:

- **train** — fit parameters on it, as often as you like.
- **validation** — choose *between* fitted models. Every choice you
  make against it spends a little of its honesty.
- **test** — one number, once, at the end. Look at it twice and you
  no longer have a test set.

---

## 4. Then transform — the minimum that works

`scikit-learn` takes a rectangle of numbers. Two things stand between
this table and one: empty cells (`temp` and `feel_temp` in runs,
`windspeed` and `weather` scattered) and four columns of words.

The minimum that fixes both: median for the numeric holes,
`"unknown"` for the categorical one, `get_dummies` for the words.
Two rules, neither optional:

- the medians are computed on **`X_train` only** — a median taken
  over rows you are about to score has seen the answer;
- every other frame is reindexed onto the training columns, so the
  matrices line up even when a category is missing from one of them.

In [ ]:
def encode(frame, medians, columns=None):
    f = frame.drop(columns=["id"]).fillna(medians)
    for c in f.columns:
        if not pd.api.types.is_numeric_dtype(f[c]):
            f[c] = f[c].fillna("unknown")
    out = pd.get_dummies(f)
    return out if columns is None else out.reindex(columns=columns, fill_value=0)


medians = X_train.median(numeric_only=True)   # training rows only

A = encode(X_train, medians)                  # the training matrix
V = encode(X_val,   medians, A.columns)
T = encode(X_test,  medians, A.columns)

print(A.shape, V.shape, T.shape)
list(A.columns)

19 columns out of 12 features. And **`hour` is still a plain
integer**, which is on purpose.

Session 2 showed that re-encoding it as 24 categories was worth about
38 MAE to a linear model, because a straight line cannot bend twice a
day. That feature is deliberately not here. Watch what a tree does
with the raw column instead.

---

## 5. Compare models — two numbers per candidate

Nine candidates: a plain line, three ridge penalties, five forests.
Each is fitted on `A` and scored twice — on the rows it was fitted to,
and on the validation rows it has never seen.

Reporting both is the habit. One of them is a measurement; the other
is a diagnosis.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error


def evaluate(name, params, model):
    model.fit(A, y_train)
    return {"model": name, "hyperparameters": params,
            "train MAE": mean_absolute_error(y_train, model.predict(A)),
            "val MAE":   mean_absolute_error(y_val,   model.predict(V))}


rows = [evaluate("LinearRegression", "—", LinearRegression())]
for alpha in (1.0, 10.0, 100.0):
    rows.append(evaluate("Ridge", f"alpha={alpha:g}", Ridge(alpha=alpha)))
for n, depth in [(100, 3), (100, 10), (100, None), (300, None), (300, 12)]:
    rows.append(evaluate("RandomForest", f"n={n}, depth={depth}",
                         RandomForestRegressor(n_estimators=n, max_depth=depth,
                                               random_state=0, n_jobs=-1)))

results = pd.DataFrame(rows)
print("predict the training mean:",
      round(mean_absolute_error(y_val, np.full(len(y_val), y_train.mean())), 2))
results.round(2)

| model | hyperparameters | train MAE | val MAE |
|---|---|---|---|
| LinearRegression | — | 79.09 | 103.78 |
| Ridge | alpha=1 | 79.08 | 103.74 |
| Ridge | alpha=10 | 79.07 | 103.43 |
| Ridge | alpha=100 | 79.16 | 102.35 |
| RandomForest | n=100, depth=3 | 64.59 | 94.75 |
| RandomForest | n=100, depth=10 | **21.58** | **65.88** |
| RandomForest | n=100, depth=None | 7.88 | 66.51 |
| RandomForest | n=300, depth=None | 7.80 | 66.82 |
| RandomForest | n=300, depth=12 | 15.37 | 66.68 |

Predicting the training mean scores 130.18 on validation, so every
row above carries signal. Read the two columns **against each other**
— neither one alone says anything.

**Regularisation buys almost nothing here.** Ridge at `alpha=100` is
1.4 MAE better than the unpenalised line, and its training error goes
*up*. That is the correct reading: with 19 columns against 8,899 rows
this model is not overfitting, it is **underfitting** — both numbers
are bad and close together. A penalty cannot fix a model that is too
simple.

**The unrestricted forest is off by 7.9 bikes an hour on training
data.** That is not skill. Every leaf holds a handful of rows and
returns their mean, so the model can recite the training set. Held
out, it scores 66.5 — nine times worse.

**And the depth-10 forest is worse on training and better on
validation** — 21.6 against 7.9, 65.9 against 66.5. Those two rows
are the whole session: the model that looks worse is the one to keep,
and the only column that could have told you is the one on the right.

**`hour` was never re-encoded.** The tree asks `hour < 7.5?`, then
`hour < 9.5?`, and cuts the day up on its own. What cost a linear
model a feature-engineering step in Session 2 is free here — it came
with the model family.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

labels = results["model"] + "  (" + results["hyperparameters"] + ")"
idx = np.arange(len(results))
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(idx - 0.2, results["train MAE"], height=0.4, color="#2f6f9f", label="train")
ax.barh(idx + 0.2, results["val MAE"], height=0.4, color="#c1553b", label="validation")
ax.set_yticks(idx)
ax.set_yticklabels(labels, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("MAE — lower is better")
ax.set_title("The gap between the bars is what the model memorised")
ax.legend()
plt.tight_layout()
plt.show()

The linear rows have two bars almost the same length; the deep-forest
rows have a stub and a full bar. That gap is memorisation, drawn.

---

## 6. Search the hyperparameters properly

Section 5 was tuning by hand: nine candidates picked by taste, ranked
on one validation slice. `GridSearchCV` does the same thing
exhaustively, and it ranks on *k* folds instead of one slice — so the
answer depends less on which 2,224 hours happened to land in
validation.

Two arguments carry the whole idea:

- **`cv=TimeSeriesSplit(n_splits=4)`.** The default `KFold` shuffles,
  which on this data trains on the future. `TimeSeriesSplit` grows the
  training window forward and always validates on hours that come
  after it.
- **`scoring="neg_mean_absolute_error"`.** scikit-learn maximises;
  the challenge ranks on −MAE. Same convention, same reason.

It fits on the **pool** — train + validation — because the search does
its own splitting inside. The slice you were choosing against by hand
goes back into the data.

48 candidates × 4 folds. Expect a few minutes on Colab.

In [ ]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

medians_pool = X_pool.median(numeric_only=True)
P      = encode(X_pool, medians_pool)
T_pool = encode(X_test, medians_pool, P.columns)

grid = {
    "n_estimators":     [200, 400],
    "max_depth":        [8, 12, 16, None],
    "min_samples_leaf": [1, 2, 5],
    "max_features":     [0.5, 1.0],
}

search = GridSearchCV(RandomForestRegressor(random_state=0, n_jobs=-1), grid,
                      cv=TimeSeriesSplit(n_splits=4),
                      scoring="neg_mean_absolute_error")
search.fit(P, y_pool)

print(len(search.cv_results_["params"]), "candidates x 4 folds")
print("best parameters:", search.best_params_)
print(f"best CV score  : {search.best_score_:.2f} (-MAE)")

In [ ]:
cv = pd.DataFrame(search.cv_results_)
cols = ["param_n_estimators", "param_max_depth", "param_min_samples_leaf",
        "param_max_features", "mean_test_score", "std_test_score"]
ranked = cv.sort_values("rank_test_score")[cols].round(2)
pd.concat([ranked.head(5), ranked.tail(3)])

The winner is `max_depth=16, max_features=1.0, min_samples_leaf=1,
n_estimators=400` at **−54.13**, and the worst candidate in the grid
is −63.09. Read what is *under* those numbers:

- **The top four are within 0.4 MAE of each other**, and
  `std_test_score` is **±14**. The folds disagree with each other
  thirty times more than the leading candidates disagree with each
  other. "The best model in the grid" is therefore a claim with a
  large error bar on it — what the search actually found is a
  *region*: depth of 12 or more, `min_samples_leaf=1`, all features
  considered at each split.
- **`max_features=0.5` is uniformly worse.** Hiding half of 19 columns
  at every split costs more than the decorrelation it buys. On a wide
  frame this argument usually earns its keep; here it does not.
- **Bigger is not better past a point.** 400 trees beat 200 by ~0.3
  MAE, for twice the compute.

The grid is the cheapest experiment in this notebook and the one most
often skipped. Run it, then read the spread — not just the winner.

---

## 7. Spend the test set

The choice is made. Now, once, refit the survivors on the pool and
score them on the hours nothing has touched.

In [ ]:
final = {
    "predict the pool mean":  np.full(len(y_test), y_pool.mean()),
    "LinearRegression":       LinearRegression().fit(P, y_pool).predict(T_pool),
    "RandomForest, depth 10": RandomForestRegressor(
        n_estimators=100, max_depth=10, random_state=0,
        n_jobs=-1).fit(P, y_pool).predict(T_pool),
    "GridSearchCV winner":    search.best_estimator_.predict(T_pool),
}

pd.DataFrame([{"model": k, "test MAE": mean_absolute_error(y_test, v)}
              for k, v in final.items()]).round(2)

| model | test MAE |
|---|---|
| predict the pool mean | 188.47 |
| LinearRegression | 146.37 |
| RandomForest, depth 10 | 52.35 |
| GridSearchCV winner | 52.77 |

**The forest is worth about 94 MAE over the line**, on hours neither
model has seen. That is the headline, and it is not marginal.

Two smaller readings, both worth more than they look:

**The grid's winner lands 0.4 MAE behind the depth-10 forest you
guessed in section 5.** That is well inside the ±14 fold spread, so
the two are indistinguishable and the search did not fail. Do **not**
now switch to the depth-10 model because it won here — choosing on
the test set is exactly the leak this whole notebook is built to
avoid, and it would leave you with no honest number at all. You chose
on cross-validation; the test set's only job was to price that
choice.

**The same forest scored 65.9 on validation and 52.4 on test.** Two
held-out slices, one model family, 13 MAE apart — because the test
model saw 2,224 more hours and because the two windows are different
months. That difference is what a single split's noise looks like,
and it is the argument for cross-validation over one slice.

---

## 8. Refit on everything, then submit

The test set has been spent, so it goes back in. Refit the chosen
configuration on all 13,903 labelled hours: the most recent of them
sit closest in time to the submission window, and they are exactly
the ones every earlier split was holding out.

In [ ]:
medians_all = X.median(numeric_only=True)
F = encode(X, medians_all)
S = encode(X_submission, medians_all, F.columns)

model = RandomForestRegressor(random_state=0, n_jobs=-1,
                              **search.best_params_).fit(F, y)
predictions = np.clip(model.predict(S), 0, None)   # no negative bicycles

submission = pd.DataFrame({"id": X_submission["id"], "prediction": predictions})
submission.to_csv("submission.csv", index=False)

assert len(submission) == len(X_submission)
assert submission["id"].is_unique
assert submission["prediction"].notna().all()
submission.head()

In [ ]:
result = client.submit(challenge_id=CHALLENGE_ID, files=["submission.csv"])
print(result)

In [ ]:
client.leaderboard(CHALLENGE_ID).head(10)

**−48.06**, against the **−138.88** your Session 2 line scored on the
same 3,476 rows. Same columns, same imputation, same encoding; only
the model and the way it was chosen changed.

It also beats the 52.8 section 7 predicted, and that is expected
rather than lucky: the final refit has the last 2,780 hours in it,
the ones nearest the submission window, which every earlier fit was
holding back.

---

## 9. Where to go from here

Everything below is chosen on validation or on CV folds. The test
slice stays untouched until you have a final answer — rebuild it from
section 3 if you want a fresh one.

**Other model families**

- **Boosting.** `HistGradientBoostingRegressor` ships with
  scikit-learn, so there is nothing to install. Trees again, but grown
  one at a time against the previous ensemble's residuals. Knobs:
  `learning_rate`, `max_iter`, `max_leaf_nodes`, `early_stopping`. On
  tabular data it is usually the strongest thing per minute of
  compute; XGBoost, LightGBM and CatBoost are the same idea with
  different defaults.
- **Support vector regression.** `SVR(kernel="rbf", C=..., gamma=...)`
  through a `StandardScaler` — it is distance-based, so unscaled
  columns break it. Fitting is roughly O(n²): on 13,903 rows expect to
  wait, and consider `LinearSVR` or a subsample first.
- **KNN.** `KNeighborsRegressor`, also after scaling. Cheap to fit,
  expensive to predict, and a good sanity check on how much local
  structure the data has.

**Better search**

- `RandomizedSearchCV` when the grid gets large — `n_iter` draws
  instead of the full product, which is usually a better use of the
  same minutes.
- Widen the grid where the winner sits on an edge. `max_depth=16` and
  `None` tied here, so depth is no longer the binding constraint.

**Features, now that the model is settled**

The forest found `hour` on its own, but it cannot invent a column:

- `ffill()` or `interpolate()` for the temperature outages — they are
  runs of consecutive hours, so the neighbouring hour is a far better
  guess than an annual median;
- an `is_missing` flag per imputed column, which lets the model use
  the fact that a reading was absent;
- `hour × workingday`, which plot 3b of the Session 2 notebook drew
  for you;
- `np.log1p(y)` as the target, then `np.expm1` on the prediction —
  standard on a skewed count.

Try one at a time and keep the ones validation agrees with. Most
engineered features do nothing, and finding that out is the skill.